In [13]:
import numpy as np
import matplotlib.pyplot as plt

In [14]:
def gravity(r1, r2, m1, m2, G=1.0):
    """Force on body 1 due to body 2 (a 2-vector)."""
    d = np.asarray(r2) - np.asarray(r1)   # points from 1 toward 2
    dist = np.linalg.norm(d)
    return G * m1 * m2 * d / dist**3

🔬 Choose smart units. Using SI units for the solar system gives numbers like 6.67e-11 and 1.99e30, which invite floating-point trouble.  Physicists rescale so G = 1.  In "solar units" (mass in solar masses, distance in AU, time in years), Newton's constant is G ≈ 4π².  We'll use G = 1 with tidy made-up masses to keep the focus on method, not bookkeeping.

In [15]:
def verlet_step(r, v, a, dt, accel_func):
    """One Velocity Verlet step. accel_func(r) returns acceleration at position r."""
    r_new = r + v * dt + 0.5 * a * dt**2
    a_new = accel_func(r_new)
    v_new = v + 0.5 * (a + a_new) * dt
    return r_new, v_new, a_new

🔬 The key trick: notice you carry the acceleration a forward between steps and reuse it, so gravity is computed only once per step despite appearing twice in the math.  Verlet is second-order accurate like Euler-Cromer, but its time-symmetry is what buys long-term stability.


In [16]:
G, M = 1.0, 1.0       # star at origin

def accel(r):
    dist = np.linalg.norm(r)
    return -G * M * r / dist**3

# Initial conditions for a bound elliptical orbit
r = np.array([1.0, 0.0])
v = np.array([0.0, 0.8])   # less than circular speed (=1.0) -> ellipse
a = accel(r)

dt = 0.001
rs = [r.copy()]
for _ in range(20000):
    r, v, a = verlet_step(r, v, a, dt, accel)
    rs.append(r.copy())
rs = np.array(rs)

✍️ Try it: The circular-orbit speed at r = 1 with GM = 1 is v = 1.  Run three cases: v0 = 1.0 (circle), v0 = 0.8 (ellipse, you're at apoapsis), and v0 = 1.5 (unbound — hyperbolic escape). Plot all three trajectories on one figure.

In [17]:
def energy(r, v, G=1.0, M=1.0, m=1.0):
    ke = 0.5 * m * np.dot(v, v)
    pe = -G * M * m / np.linalg.norm(r)
    return ke + pe

def ang_momentum(r, v, m=1.0):
    # 2D cross product r x v is a scalar
    return m * (r[0] * v[1] - r[1] * v[0])

✍️ Before the lab: Refactor your verlet_step so it takes the acceleration function as an argument.  In Week 7 your engine will swap integrators the way Lab 2 swapped Euler for Euler-Cromer, keeping the integrator decoupled from the force is the whole game.